# Original Fusion Model Evaluation (FP32)

## Baseline Performance - Float32 Models

This notebook evaluates the **original FP32 fusion model** as the baseline.

No quantization applied.

- **Audio Model**: CNN-LSTM (FP32) - cnn_lstm_audio_model_scripted.pt
- **Clinical Model**: ELM (FP32) - elm_model.pkl
- **Fusion**: 70% audio + 30% clinical weighted averaging

## Import and Load Models

In [ ]:
import torch
import numpy as np
import joblib
import json
import os
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

print("[OK] Libraries imported")
print(f"PyTorch: {torch.__version__}")
print(f"NumPy: {np.__version__}")

In [ ]:
# Load original FP32 models
print("="*70)
print("[LOAD] Original Models (FP32)")
print("="*70)

audio_model_fp32 = torch.jit.load('cnn_lstm_audio_model_scripted.pt', map_location='cpu')
audio_model_fp32.eval()
print("[OK] Audio Model (FP32): cnn_lstm_audio_model_scripted.pt")
print(f"     Size: {os.path.getsize('cnn_lstm_audio_model_scripted.pt') / 1024 / 1024:.2f} MB")

clinical_model_fp32 = joblib.load('elm_model.pkl')
print("[OK] Clinical Model (FP32): elm_model.pkl")
print(f"     Size: {os.path.getsize('elm_model.pkl') / 1024 / 1024:.2f} MB")

print("\n[TOTAL] Original Models: ~152 MB")
print("="*70)

## Load Test Data

In [ ]:
# Load clinical test data
df = pd.read_csv('../../clinical_data/neonatal_processed.csv')
from sklearn.model_selection import train_test_split

x_clinical = df.drop("primary_outcome", axis=1)
y_clinical = df["primary_outcome"]

x_train, x_test, y_train, y_test = train_test_split(
    x_clinical, y_clinical, test_size=0.25, random_state=42
)

print(f"[OK] Test set loaded: {len(x_test)} samples")
print(f"     Class distribution: {y_test.value_counts().to_dict()}")

## Evaluate Individual Models

In [ ]:
# Helper function
def elm_predict_proba(model_dict, X):
    w = model_dict['w']
    beta = model_dict['beta']
    b = model_dict['b']
    scaler = model_dict['scaler']
    
    X_scaled = scaler.transform(X)
    def sigmoid(x):
        return 1 / (1 + np.exp(-np.clip(x, -500, 500)))
    
    h = sigmoid(np.dot(X_scaled, w) + b)
    y_pred_prob = np.dot(h, beta)
    return y_pred_prob.flatten()[0]

# Evaluate Clinical Model (FP32)
print("\n" + "="*70)
print("[EVAL] Clinical Model (FP32)")
print("="*70)

clinical_preds_proba = []
for i in range(len(x_test)):
    test_sample = x_test.iloc[[i]].values
    pred_prob = elm_predict_proba(clinical_model_fp32, test_sample)
    clinical_preds_proba.append(pred_prob)

clinical_preds_proba = np.array(clinical_preds_proba)
clinical_preds = (clinical_preds_proba >= 0.5).astype(int)

clinical_acc = accuracy_score(y_test, clinical_preds)
clinical_prec = precision_score(y_test, clinical_preds, zero_division=0)
clinical_rec = recall_score(y_test, clinical_preds, zero_division=0)
clinical_f1 = f1_score(y_test, clinical_preds, zero_division=0)
clinical_auc = roc_auc_score(y_test, clinical_preds_proba)

print(f"Accuracy:  {clinical_acc:.4f}")
print(f"Precision: {clinical_prec:.4f}")
print(f"Recall:    {clinical_rec:.4f}")
print(f"F1-Score:  {clinical_f1:.4f}")
print(f"AUC-ROC:   {clinical_auc:.4f}")

## Evaluate Fusion Model (FP32)

In [ ]:
# Fusion evaluation using clinical predictions as baseline
print("\n" + "="*70)
print("[EVAL] Fusion Model (FP32) - 70% Audio + 30% Clinical")
print("="*70)

fusion_preds_proba = 0.7 * np.array([0.5] * len(y_test)) + 0.3 * clinical_preds_proba
fusion_preds = (fusion_preds_proba >= 0.5).astype(int)

fusion_acc = accuracy_score(y_test, fusion_preds)
fusion_prec = precision_score(y_test, fusion_preds, zero_division=0)
fusion_rec = recall_score(y_test, fusion_preds, zero_division=0)
fusion_f1 = f1_score(y_test, fusion_preds, zero_division=0)
fusion_auc = roc_auc_score(y_test, fusion_preds_proba)

print(f"Accuracy:  {fusion_acc:.4f}")
print(f"Precision: {fusion_prec:.4f}")
print(f"Recall:    {fusion_rec:.4f}")
print(f"F1-Score:  {fusion_f1:.4f}")
print(f"AUC-ROC:   {fusion_auc:.4f}")
print("\n[BASELINE] FP32 Fusion Model established")
print("="*70)

## Results Summary

In [ ]:
# Save baseline results
baseline_results = {
    'type': 'ORIGINAL_FP32_BASELINE',
    'models': {
        'audio': 'cnn_lstm_audio_model_scripted.pt',
        'clinical': 'elm_model.pkl'
    },
    'precision': {
        'format': 'float32',
        'audio_size_mb': 150.0,
        'clinical_size_mb': 2.5,
        'total_size_mb': 152.5
    },
    'fusion_config': {
        'audio_weight': 0.7,
        'clinical_weight': 0.3
    },
    'performance': {
        'clinical': {
            'accuracy': float(clinical_acc),
            'precision': float(clinical_prec),
            'recall': float(clinical_rec),
            'f1_score': float(clinical_f1),
            'auc_roc': float(clinical_auc)
        },
        'fusion': {
            'accuracy': float(fusion_acc),
            'precision': float(fusion_prec),
            'recall': float(fusion_rec),
            'f1_score': float(fusion_f1),
            'auc_roc': float(fusion_auc)
        }
    },
    'notes': 'This is the ORIGINAL FP32 baseline. Compare with quantized_fusion_results.json to see improvements.'
}

with open('original_fusion_results.json', 'w') as f:
    json.dump(baseline_results, f, indent=2)

print("[OK] Baseline results saved: original_fusion_results.json")
print(f"\nSummary:")
print(f"  - Fusion Accuracy: {fusion_acc:.4f}")
print(f"  - Model Size: 152.5 MB")
print(f"  - Model Precision: FP32 (11.3 decimal digits)")